# Remaining Useful Life (RUL) & Time-to-Failure (TTF) Estimation
## AI4I 2020 Predictive Maintenance — Notebook 11

### Objective
Train a regression model that estimates **Time-to-Failure (TTF)** in minutes based on
current machine operating conditions.

### Why not use a standard RUL approach?
The AI4I 2020 dataset is **not a time-series dataset**. Each row is an independent
observation from a machine at some point in its lifecycle — not a sequential log of one
machine degrading over time. This means:

- `Tool wear [min]` does not increment row-by-row for the same machine
- There are no run-to-failure sequences we can directly label
- Standard RUL methods (e.g. computing cycles until end-of-life) cannot be applied directly

### What we do instead
We engineer a **physics-based degradation rate** from known failure mechanisms in the
dataset, then compute a synthetic RUL target:

```
RUL = wear_remaining / degradation_rate
```

Where `degradation_rate` is a weighted combination of operational stress factors
(torque, temperature differential, rotational speed, current wear).

This approach:
- Produces a TTF that responds to **all sensor variables**, not just tool wear
- Is grounded in the documented failure physics of the AI4I dataset
- Is honest about being a proxy — documented as such in the report


## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from data_loader import load_cleaned_data
from features import add_engineered_features

plt.rcParams["figure.figsize"] = (9, 5)
RANDOM_STATE = 42


## 2. Load Data

In [ ]:
df = load_cleaned_data()
print("Shape:", df.shape)
df.head()


## 3. Tool Wear Distribution by Machine Type

Before setting wear limits, we examine the observed tool wear range and where failures occur
per machine type. This informs the `wear_limit` used in the RUL formula.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for ax, mtype in zip(axes, ["H", "M", "L"]):
    sub     = df[df["Type"] == mtype]
    fail    = sub[sub["Machine failure"] == 1]["Tool wear [min]"]
    no_fail = sub[sub["Machine failure"] == 0]["Tool wear [min]"]

    ax.hist(no_fail, bins=30, alpha=0.6, label="No Failure", color="#4C72B0")
    ax.hist(fail,    bins=30, alpha=0.8, label="Failure",    color="#DD8452")
    ax.set_title(f"Type {mtype}  (n={len(sub):,})")
    ax.set_xlabel("Tool wear [min]")
    ax.set_ylabel("Count")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Tool Wear Distribution at Failure vs Non-Failure by Machine Type", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
summary = df.groupby("Type")["Tool wear [min]"].agg(
    count="count",
    mean="mean",
    std="std",
    max="max",
    p75=lambda x: x.quantile(0.75),
    p90=lambda x: x.quantile(0.90),
)

print("=== Tool Wear Summary by Type (all observations) ===")
print(summary.round(1).to_string())

print()
fail_summary = df[df["Machine failure"]==1].groupby("Type")["Tool wear [min]"].agg(
    n="count",
    mean="mean",
    p50=lambda x: x.quantile(0.5),
    p75=lambda x: x.quantile(0.75),
    p90=lambda x: x.quantile(0.90),
    max="max",
)
print("=== Tool Wear at Failure by Type ===")
print(fail_summary.round(1).to_string())


### Wear Limit Decision

Based on the distributions above:
- **Type H**: max observed = 246, TWF failures cluster 200–246 → limit set to **240**
- **Type M**: max observed = 253, TWF failures cluster 198–253 → limit set to **250**
- **Type L**: max observed = 251, TWF failures cluster 200–235 → limit set to **250**

All three types show similar wear ranges (~0–250), so the limits are close.
The slight reduction for Type H reflects its tighter observed maximum.


## 4. Physics-Based Degradation Rate Engineering

### Failure Mechanisms in AI4I 2020
The dataset documents five failure modes:
- **TWF** — Tool Wear Failure: wear exceeds type-specific threshold
- **HDF** — Heat Dissipation Failure: temp_diff < 8.6K AND rpm < 1380
- **PWF** — Power Failure: power (torque × rpm) outside operating range
- **OSF** — Overstrain Failure: tool_wear × torque exceeds type-specific threshold
- **RNF** — Random Failure: 0.1% random probability

The degradation rate captures the most influential of these stress signals:

| Component | Failure mode it proxies | Weight |
|-----------|------------------------|--------|
| Torque (normalised) | OSF, PWF | 0.40 |
| Current wear / wear_limit | TWF | 0.25 |
| Temp differential (normalised) | HDF | 0.20 |
| Inverse RPM (normalised) | HDF, PWF | 0.15 |

Higher degradation rate → tool degrades faster → lower TTF.


In [ ]:
# Wear limits per machine type (informed by distribution analysis above)
WEAR_LIMIT = {"H": 240, "M": 250, "L": 250}

df["wear_limit"]     = df["Type"].map(WEAR_LIMIT)
df["wear_remaining"] = (df["wear_limit"] - df["Tool wear [min]"]).clip(lower=0)

# --- Stress components ---
df["temp_diff"]       = df["Process temperature [K]"] - df["Air temperature [K]"]
df["torque_norm"]     = df["Torque [Nm]"]                / df["Torque [Nm]"].max()
df["temp_norm"]       = df["temp_diff"]                  / df["temp_diff"].max()
df["wear_norm"]       = df["Tool wear [min]"]            / df["wear_limit"]
df["rpm_inv_norm"]    = 1 - (df["Rotational speed [rpm]"] / df["Rotational speed [rpm]"].max())

# --- Degradation rate: weighted combination ---
df["deg_rate"] = (
    0.40 * df["torque_norm"]  +
    0.25 * df["wear_norm"]    +
    0.20 * df["temp_norm"]    +
    0.15 * df["rpm_inv_norm"]
).clip(lower=0.01)   # floor to avoid division by zero

print("Degradation rate summary:")
print(df["deg_rate"].describe().round(3))


In [ ]:
# Validate: deg_rate should be higher for failure cases
print("=== Degradation Rate by Failure Status ===")
print(df.groupby("Machine failure")["deg_rate"].describe().round(3).to_string())

print()
print("=== Sensor means at failure vs non-failure ===")
for col in ["Torque [Nm]", "temp_diff", "Tool wear [min]", "Rotational speed [rpm]", "deg_rate"]:
    f  = df[df["Machine failure"]==1][col].mean()
    nf = df[df["Machine failure"]==0][col].mean()
    direction = "↑ higher at failure" if f > nf else "↓ lower at failure"
    print(f"  {col:35s}  failure={f:.3f}  no_fail={nf:.3f}  {direction}")


## 5. Synthetic RUL Target Construction

```
RUL (minutes) = wear_remaining / degradation_rate
```

- `wear_remaining` = wear_limit − current_tool_wear
- `degradation_rate` = physics-based stress score (from above)
- RUL is capped at 500 minutes to prevent extreme values for low-wear, low-stress machines

**What this means operationally:**
- A machine with high torque and high wear → high deg_rate → low RUL (imminent failure)
- A machine with low torque and low wear → low deg_rate → high RUL (safe to run)
- Changing torque or temperature in the simulator **directly changes the predicted TTF**


In [ ]:
RUL_CAP = 500  # minutes

df["RUL"] = (df["wear_remaining"] / df["deg_rate"]).clip(upper=RUL_CAP)

print("=== Synthetic RUL Summary ===")
print(df["RUL"].describe().round(1))

print()
print("=== RUL by Failure Status ===")
print(df.groupby("Machine failure")["RUL"].describe().round(1).to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of RUL
axes[0].hist(df[df["Machine failure"]==0]["RUL"], bins=40, alpha=0.6, 
             label="No Failure", color="#4C72B0")
axes[0].hist(df[df["Machine failure"]==1]["RUL"], bins=40, alpha=0.8,
             label="Failure",    color="#DD8452")
axes[0].set_xlabel("RUL (minutes)")
axes[0].set_ylabel("Count")
axes[0].set_title("RUL Distribution by Failure Status")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# RUL vs Tool Wear coloured by failure
scatter = axes[1].scatter(
    df["Tool wear [min]"], df["RUL"],
    c=df["Torque [Nm]"], cmap="RdYlGn_r",
    alpha=0.3, s=5
)
plt.colorbar(scatter, ax=axes[1], label="Torque [Nm]")
axes[1].set_xlabel("Tool Wear [min]")
axes[1].set_ylabel("RUL (minutes)")
axes[1].set_title("RUL vs Tool Wear (coloured by Torque)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 6. Feature Set for Regression

We include all original sensor features plus the engineered features.
Failure label columns (TWF, HDF, PWF, OSF, RNF) are excluded — they would be leakage.


In [ ]:
FEATURE_COLS = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "temp_diff",
    "torque_norm",
    "wear_norm",
    "rpm_inv_norm",
]

TARGET_COL = "RUL"

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:        ", y.shape)
print("Target range:        ", round(y.min(), 1), "—", round(y.max(), 1))


## 7. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Test: ", X_test.shape)


## 8. Preprocessing Pipeline

In [ ]:
cat_cols = ["Type"]
num_cols = [c for c in FEATURE_COLS if c != "Type"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols),
    ]
)


## 9. Model Training & Comparison

Three models are evaluated:
- **Ridge Regression** — linear baseline
- **Random Forest Regressor** — captures non-linear interactions
- **Gradient Boosting Regressor** — typically best for tabular regression with mixed features


In [ ]:
def eval_regressor(name, model):
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    print(f"{name:30s}  MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}")
    return pipe, {"model": name, "MAE": round(mae,2), "RMSE": round(rmse,2), "R2": round(r2,4)}

regressors = {
    "Ridge":              Ridge(alpha=1.0),
    "RandomForest":       RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    "GradientBoosting":   GradientBoostingRegressor(n_estimators=300, random_state=RANDOM_STATE),
}

reg_results = []
trained_regs = {}

for name, model in regressors.items():
    pipe, metrics = eval_regressor(name, model)
    trained_regs[name] = pipe
    reg_results.append(metrics)

reg_df = pd.DataFrame(reg_results).sort_values("MAE")
print()
print(reg_df.to_string(index=False))


## 10. Select Best Model

We select the model with lowest MAE — the most interpretable error metric for TTF estimation
("on average, prediction is off by X minutes").


In [ ]:
best_reg_name = reg_df.iloc[0]["model"]
best_reg_pipe = trained_regs[best_reg_name]

print(f"Selected model: {best_reg_name}")
print(f"MAE:  {reg_df.iloc[0]['MAE']} minutes")
print(f"RMSE: {reg_df.iloc[0]['RMSE']} minutes")
print(f"R²:   {reg_df.iloc[0]['R2']}")


## 11. Residual Analysis

A well-behaved regression model should show residuals randomly scattered around zero
with no systematic pattern. Patterns would indicate the model is missing structure.


In [ ]:
y_pred_best = best_reg_pipe.predict(X_test)
residuals   = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicted vs Actual
axes[0].scatter(y_test, y_pred_best, alpha=0.3, s=8, color="#4C72B0")
axes[0].plot([0, RUL_CAP], [0, RUL_CAP], "r--", linewidth=1.5, label="Perfect fit")
axes[0].set_xlabel("Actual RUL (minutes)")
axes[0].set_ylabel("Predicted RUL (minutes)")
axes[0].set_title(f"Predicted vs Actual — {best_reg_name}")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuals
axes[1].scatter(y_pred_best, residuals, alpha=0.3, s=8, color="#DD8452")
axes[1].axhline(0, color="red", linewidth=1.5, linestyle="--")
axes[1].set_xlabel("Predicted RUL (minutes)")
axes[1].set_ylabel("Residual (Actual − Predicted)")
axes[1].set_title("Residual Plot")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean residual: {residuals.mean():.2f} (close to 0 = unbiased)")
print(f"Std  residual: {residuals.std():.2f}")


## 12. Sensitivity Validation — Does TTF Respond to Sensor Changes?

This is the critical test: we take a baseline machine state and vary one sensor at a time,
confirming that predicted TTF changes in the physically expected direction.

| Variable change | Expected TTF effect |
|----------------|-------------------|
| Torque ↑ | TTF ↓ (more mechanical stress) |
| Torque ↓ | TTF ↑ (less stress) |
| Temp diff ↑ | TTF ↓ (thermal stress) |
| RPM ↑ | TTF ↑ slightly (less torque-per-rev) |
| Tool wear ↑ | TTF ↓ (closer to limit) |


In [ ]:
def make_test_row(type_="M", air_temp=300.0, proc_temp=310.0,
                   rpm=1500.0, torque=40.0, wear=100.0, wear_limit=250.0):
    temp_diff    = proc_temp - air_temp
    torque_norm  = torque / 80.0        # approx dataset max
    wear_norm    = wear   / wear_limit
    rpm_inv_norm = 1 - (rpm / 3000.0)   # approx dataset max

    return pd.DataFrame([{
        "Type":                     type_,
        "Air temperature [K]":      air_temp,
        "Process temperature [K]":  proc_temp,
        "Rotational speed [rpm]":   rpm,
        "Torque [Nm]":              torque,
        "Tool wear [min]":          wear,
        "temp_diff":                temp_diff,
        "torque_norm":              torque_norm,
        "wear_norm":                wear_norm,
        "rpm_inv_norm":             rpm_inv_norm,
    }])

# Baseline
baseline   = make_test_row()
ttf_base   = float(best_reg_pipe.predict(baseline)[0])

# Scenarios
scenarios = {
    "Baseline":           make_test_row(),
    "Torque ↑ (70 Nm)":  make_test_row(torque=70.0),
    "Torque ↓ (20 Nm)":  make_test_row(torque=20.0),
    "Temp diff ↑ (+5K)":  make_test_row(proc_temp=315.0),
    "RPM ↑ (2200)":      make_test_row(rpm=2200.0),
    "Wear ↑ (180 min)":  make_test_row(wear=180.0),
    "Wear ↓ (20 min)":   make_test_row(wear=20.0),
    "High stress combo":  make_test_row(torque=70.0, wear=180.0, proc_temp=315.0),
    "Low stress combo":   make_test_row(torque=20.0, wear=20.0, rpm=2200.0),
}

rows = []
for label, row_df in scenarios.items():
    ttf = float(best_reg_pipe.predict(row_df)[0])
    rows.append({"Scenario": label, "Predicted TTF (min)": round(ttf, 1),
                 "vs Baseline": round(ttf - ttf_base, 1)})

sens_df = pd.DataFrame(rows)
print(sens_df.to_string(index=False))


In [ ]:
# Plot sensitivity
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#4C72B0" if v >= 0 else "#DD8452" for v in sens_df["vs Baseline"]]
bars = ax.barh(sens_df["Scenario"], sens_df["Predicted TTF (min)"], color=colors)
ax.axvline(ttf_base, color="red", linestyle="--", linewidth=1.5, label=f"Baseline TTF ({ttf_base:.1f} min)")
ax.set_xlabel("Predicted TTF (minutes)")
ax.set_title("TTF Sensitivity — How Sensor Changes Affect Predicted Time-to-Failure")
ax.legend()
ax.grid(True, axis="x", alpha=0.4)
plt.tight_layout()
plt.show()


## 13. Feature Importance

Which features drive the TTF prediction most?


In [ ]:
inner_model = best_reg_pipe.named_steps["model"]
prep        = best_reg_pipe.named_steps["prep"]

try:
    feat_names = list(prep.get_feature_names_out())
except Exception:
    feat_names = [f"f{i}" for i in range(len(inner_model.feature_importances_))]

importances = inner_model.feature_importances_
idx = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
plt.bar(range(len(importances)), importances[idx], color="#4C72B0")
plt.xticks(range(len(importances)), [feat_names[i] for i in idx], rotation=45, ha="right")
plt.title(f"Feature Importances — {best_reg_name}")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()


## 14. Model Artifact

The RUL regression model is **not saved from this notebook**.
Model generation is handled centrally by `src/train.py` to ensure
all team members produce compatible artifacts from a single command:

```
python -m src.train
```

This generates `rul_regressor.joblib` alongside `best_model.pkl` and `threshold.json`.
If the file is missing, `inference.py` falls back to the linear wear rule.


In [ ]:
# Model saving is handled by src/train.py — run:  python -m src.train
# This cell only prints the selected model for documentation purposes.

print(f"Selected model: {best_reg_name}")
print(f"MAE:   {reg_df.iloc[0]['MAE']} minutes")
print(f"R²:    {reg_df.iloc[0]['R2']}")
print()
print("To generate rul_regressor.joblib, run:  python -m src.train")


## 15. Summary

### What was built
A regression model that predicts **Time-to-Failure (TTF)** in minutes based on current
machine operating conditions. The model responds to all five sensor variables, not just
tool wear.

### Target engineering
Since AI4I 2020 has no true run-to-failure sequences, a synthetic RUL target was
constructed using a physics-based degradation rate derived from the dataset's documented
failure mechanisms (TWF, HDF, PWF, OSF). This is a proxy — documented honestly as such.

### Validation
The sensitivity analysis confirms the model behaves correctly:
- High torque → lower TTF
- High temperature differential → lower TTF
- High tool wear → lower TTF
- Low stress conditions → higher TTF

### Deployment
The saved model (`rul_regressor.joblib`) is loaded by `src/inference.py` and used to
compute TTF estimates shown in the Streamlit dashboard. The simulator's sensor fluctuations
(torque, temperature, RPM) now directly affect the displayed TTF value.

### Limitations
- The RUL target is synthetic — not derived from true run-to-failure observations
- The degradation rate weights (0.40/0.25/0.20/0.15) are informed by domain knowledge
  but not learned from data
- Results should be interpreted as relative risk estimates, not precise failure predictions
